# Capstone 1 - Twitter Customer Support Routing (SOLUTION)

*Module 1 capstone - ML & NLP by Data Trainers LLC.*

This is the complete solution notebook. It mirrors the exercise structure, fills in every `None  # YOUR CODE` block, and adds the extras called out in the plan:

- Class-distribution plot (exercise has this too).
- Full PyTorch model + training loop with manual early stopping.
- Confusion matrix for both classical and neural.
- Train/val loss curves.
- Misclassification analysis: a few wrong predictions per class with discussion.
- Side-by-side summary of LogReg vs. neural, plus a recommendation on which to ship.

## Problem recap

We route inbound customer tweets at a support team to the right company queue (AppleSupport / AmazonHelp / SpotifyCares / Uber_Support). Labels come from joining each inbound tweet to its reply and using the reply author's handle. The data is class-imbalanced. We compare TF-IDF + LogReg against a small PyTorch Embedding + MLP.

## Section 0: Environment Setup

In [ ]:
# Install required packages (skip if you already have them locally)
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn textblob

In [ ]:
# Core imports
import os, re, random, math, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")

## Section 1: Load TWCS

In [ ]:
%%writefile get_capstone_data.sh
if [ ! -f twcs.csv ]; then
  wget -O twcs.csv "https://www.dropbox.com/scl/fi/6w2dchvsthwdol934nprt/twcs.csv?rlkey=u5205iehhrog88qt8iwm4udxs&dl=1"
fi

In [ ]:
!bash get_capstone_data.sh

In [ ]:
SUBSET_SIZE = 100_000
twcs = pd.read_csv('./twcs.csv', nrows=SUBSET_SIZE)
print(f"Loaded {len(twcs):,} tweets")
twcs.head()

In [ ]:
print(twcs['inbound'].value_counts())
print()
print(twcs['author_id'].value_counts().head(10))

## Section 2: Extract Labels

Join each inbound tweet to the reply whose `in_response_to_tweet_id` matches its `tweet_id`. The reply's `author_id` is the label.

In [ ]:
TOP_AUTHORS = ['AppleSupport', 'AmazonHelp', 'SpotifyCares', 'Uber_Support']
print(f"Target classes: {TOP_AUTHORS}")

In [ ]:
inbound = twcs[twcs['inbound'] == True].copy()
outbound = twcs[twcs['inbound'] == False].copy()
print(f"inbound:  {len(inbound):,}")
print(f"outbound: {len(outbound):,}")

In [ ]:
outbound_targeted = outbound[outbound['author_id'].isin(TOP_AUTHORS)]
outbound_targeted = outbound_targeted.dropna(subset=['in_response_to_tweet_id'])
outbound_targeted['in_response_to_tweet_id'] = outbound_targeted['in_response_to_tweet_id'].astype('int64')

reply_map = dict(zip(outbound_targeted['in_response_to_tweet_id'], outbound_targeted['author_id']))
print(f"reply_map entries: {len(reply_map):,}")
print("sample:", dict(list(reply_map.items())[:3]))

In [ ]:
inbound['label'] = inbound['tweet_id'].map(reply_map)
labeled = inbound.dropna(subset=['label']).copy()
labeled = labeled[['tweet_id', 'text', 'label']].reset_index(drop=True)
print(f"Labeled tweets: {len(labeled):,}")
labeled.head()

In [ ]:
class_counts = labeled['label'].value_counts()
print(class_counts)

plt.figure(figsize=(7, 3.5))
class_counts.plot(kind='bar', color=['#5B8DEF', '#FF9F40', '#4ECDC4', '#FFD166'])
plt.title('Tweets per target company (before balancing)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Lab 1 solution: cap per-class size at 25K

In [ ]:
PER_CLASS_CAP = 25_000

# Groupby + sample, capping at PER_CLASS_CAP (or class size if smaller).
balanced = (
    labeled.groupby('label', group_keys=False)
           .apply(lambda g: g.sample(n=min(len(g), PER_CLASS_CAP), random_state=SEED))
)
# Shuffle the combined rows so classes are interleaved.
balanced = balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Balanced dataset size: {len(balanced):,}")
print(balanced['label'].value_counts())

## Section 3: Clean the Tweets

We strip URLs and `@handles` (handles would leak the label) and keep the `#` character so hashtags stay meaningful as tokens.

In [ ]:
URL_RE = re.compile(r'http\S+|www\.\S+|t\.co/\S+')
HANDLE_RE = re.compile(r'@\w+')
NON_ALPHA_RE = re.compile(r"[^a-z0-9#\s]")
WS_RE = re.compile(r'\s+')

def clean_tweet(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = URL_RE.sub(' ', text)
    text = HANDLE_RE.sub(' ', text)
    text = NON_ALPHA_RE.sub(' ', text)
    text = WS_RE.sub(' ', text).strip()
    return text

for ex in balanced['text'].iloc[:3]:
    print('RAW  :', ex)
    print('CLEAN:', clean_tweet(ex))
    print()

### Lab 2 solution: cleaning + stratified train/val/test split

In [ ]:
# 1. Clean every tweet
balanced['clean'] = balanced['text'].apply(clean_tweet)

# 2. Drop empty cleaned tweets (some tweets were all URLs/handles)
balanced = balanced[balanced['clean'].str.len() > 0].reset_index(drop=True)
print(f"After dropping empties: {len(balanced):,}")

# 3. Stratified 80/10/10 split. First carve off 10% test, then 1/9 of the rest for val (= 10% of original).
X_all, y_all = balanced['clean'].values, balanced['label'].values
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.10, stratify=y_all, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=1/9, stratify=y_trainval, random_state=SEED
)

print(f"Train: {len(X_train):,}   Val: {len(X_val):,}   Test: {len(X_test):,}")
print("Train class distribution:", pd.Series(y_train).value_counts().to_dict())

## Section 4: Classical Baseline - TF-IDF + Logistic Regression

The baseline to beat.

In [ ]:
# 1. Vectorize: unigrams + bigrams, drop words that appear in <3 docs.
tfidf = TfidfVectorizer(min_df=3, ngram_range=(1, 2), sublinear_tf=True)
Xtr_tfidf  = tfidf.fit_transform(X_train)
Xval_tfidf = tfidf.transform(X_val)
Xte_tfidf  = tfidf.transform(X_test)
print(f"TF-IDF matrix: {Xtr_tfidf.shape}")

# 2. Train LogReg with balanced class weights
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1, random_state=SEED)
logreg.fit(Xtr_tfidf, y_train)

# 3. Predict on test
y_pred_lr = logreg.predict(Xte_tfidf)
print(classification_report(y_test, y_pred_lr, digits=3))
lr_macro_f1 = f1_score(y_test, y_pred_lr, average='macro')
print(f"LogReg test macro-F1: {lr_macro_f1:.4f}")

### Lab 4 solution: confusion matrix for the baseline

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=TOP_AUTHORS)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=TOP_AUTHORS, yticklabels=TOP_AUTHORS, cbar=False)
plt.title(f'LogReg - confusion matrix (macro-F1 = {lr_macro_f1:.3f})')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

### Top words per class (interpretability)

Because LogReg is linear, we can peek at the most influential words for each class by sorting `coef_[i]`.

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
for class_idx, class_name in enumerate(logreg.classes_):
    top_ids = np.argsort(logreg.coef_[class_idx])[-10:][::-1]
    print(f"{class_name:15s} -> {', '.join(feature_names[top_ids])}")

## Section 5: Neural Classifier in PyTorch

In [ ]:
# Hyperparameters
VOCAB_MIN_FREQ = 3
MAX_LEN        = 40
EMBEDDING_DIM  = 64
HIDDEN_DIM     = 128
BATCH_SIZE     = 64
EPOCHS         = 10
LR             = 1e-3
DROPOUT        = 0.3
PATIENCE       = 2

PAD_IDX, UNK_IDX = 0, 1

def tokenize(text):
    return text.split()

def build_vocab(texts, min_freq=VOCAB_MIN_FREQ):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {'<pad>': PAD_IDX, '<unk>': UNK_IDX}
    for token, freq in counter.most_common():
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size:,}")

In [ ]:
LABEL2ID = {name: i for i, name in enumerate(TOP_AUTHORS)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}

def encode_text(text, max_len=MAX_LEN):
    return [vocab.get(tok, UNK_IDX) for tok in tokenize(text)[:max_len]]

y_train_ids = np.array([LABEL2ID[l] for l in y_train])
y_val_ids   = np.array([LABEL2ID[l] for l in y_val])
y_test_ids  = np.array([LABEL2ID[l] for l in y_test])
print("Train id counts:", np.bincount(y_train_ids))

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [encode_text(t) for t in texts]
        self.labels = labels.astype(np.int64)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])

def pad_collate(batch):
    seqs, ys = zip(*batch)
    maxlen = max(1, max(len(s) for s in seqs))
    ids = torch.full((len(seqs), maxlen), PAD_IDX, dtype=torch.long)
    mask = torch.zeros((len(seqs), maxlen), dtype=torch.long)
    for i, s in enumerate(seqs):
        if len(s) == 0:
            ids[i, 0] = UNK_IDX
            mask[i, 0] = 1
        else:
            ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
            mask[i, :len(s)] = 1
    y = torch.tensor(ys, dtype=torch.long)
    return ids, mask, y

train_ds = TweetDataset(X_train, y_train_ids)
val_ds   = TweetDataset(X_val,   y_val_ids)
test_ds  = TweetDataset(X_test,  y_test_ids)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=pad_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate)

ids_b, mask_b, y_b = next(iter(train_loader))
print("batch ids shape:", ids_b.shape, "mask:", mask_b.shape, "y:", y_b.shape)

### Lab 5 solution: TweetClassifier

Full implementation with masked mean-pool. Note `padding_idx=PAD_IDX`: that keeps the embedding for padding exactly zero and stops gradients from flowing into it, which pairs well with the masked mean.

In [ ]:
class TweetClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_classes, pad_idx=PAD_IDX, dropout=DROPOUT):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.fc1  = nn.Linear(emb_dim, hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, mask):
        # x: (B, T); mask: (B, T) with 1 at real tokens, 0 at padding
        e = self.emb(x)                                          # (B, T, E)
        mask_f = mask.unsqueeze(-1).float()                      # (B, T, 1)
        e_mean = (e * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)  # (B, E)
        h = torch.relu(self.fc1(e_mean))                         # (B, H)
        return self.fc2(self.drop(h))                            # (B, n_classes) - logits

model = TweetClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, n_classes=len(TOP_AUTHORS)).to(device)
print(model)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Class weights inverse to frequency, matching sklearn's 'balanced' recipe.
class_counts_train = np.bincount(y_train_ids, minlength=len(TOP_AUTHORS))
class_weights = len(y_train_ids) / (len(TOP_AUTHORS) * class_counts_train)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
print("Class counts (train):", class_counts_train.tolist())
print("Class weights        :", np.round(class_weights, 3).tolist())

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, targs = [], []
    with torch.no_grad():
        for ids, mask, y in loader:
            ids, mask = ids.to(device), mask.to(device)
            logits = model(ids, mask)
            preds.append(logits.argmax(dim=-1).cpu().numpy())
            targs.append(y.numpy())
    y_hat  = np.concatenate(preds)
    y_true = np.concatenate(targs)
    return f1_score(y_true, y_hat, average='macro'), y_hat, y_true

### Lab 6 solution: full training loop with manual early stopping

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn   = nn.CrossEntropyLoss(weight=class_weights_t)

best_val_f1 = 0.0
patience_counter = 0
best_state = None
history = {'train_loss': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    # ---- train phase ----
    model.train()
    total_loss, n_batches = 0.0, 0
    for ids, mask, y in train_loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    train_loss = total_loss / max(1, n_batches)

    # ---- validation phase ----
    val_f1, _, _ = evaluate(model, val_loader)

    history['train_loss'].append(train_loss)
    history['val_f1'].append(val_f1)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_macroF1={val_f1:.4f}")

    # ---- manual early stopping ----
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
            break

print(f"Best val macro-F1: {best_val_f1:.4f}")

### Train/val curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history['train_loss'], marker='o')
axes[0].set_title('Train loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(history['val_f1'], marker='o', color='#FF7F50')
axes[1].set_title('Validation macro-F1')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
plt.tight_layout()
plt.show()

### Lab 7 solution: evaluate on test

In [ ]:
# Restore best weights before scoring on test
if best_state is not None:
    model.load_state_dict(best_state)

test_f1, y_pred_nn, y_true_nn = evaluate(model, test_loader)
print(f"Neural model test macro-F1: {test_f1:.4f}\n")

target_names = [ID2LABEL[i] for i in range(len(TOP_AUTHORS))]
print(classification_report(y_true_nn, y_pred_nn, target_names=target_names, digits=3))

cm_nn = confusion_matrix(y_true_nn, y_pred_nn, labels=list(range(len(TOP_AUTHORS))))
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names, cbar=False)
plt.title(f'Neural - confusion matrix (macro-F1 = {test_f1:.3f})')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()

### Extra: misclassification analysis

Pull a few wrong predictions from each true class and eyeball them. Often the model gets confused on short, ambiguous tweets ("help please!!!") that carry no topical signal.

In [ ]:
wrong_idx = np.where(y_pred_nn != y_true_nn)[0]
print(f"Total misclassifications: {len(wrong_idx):,} / {len(y_true_nn):,}")

# Sample up to 3 misclassifications per true class
rng = np.random.default_rng(SEED)
for true_cls in range(len(TOP_AUTHORS)):
    ids_of_class = [i for i in wrong_idx if y_true_nn[i] == true_cls]
    take = rng.choice(ids_of_class, size=min(3, len(ids_of_class)), replace=False) if ids_of_class else []
    print(f"\n=== True = {ID2LABEL[true_cls]} ===")
    for i in take:
        pred_cls = ID2LABEL[int(y_pred_nn[i])]
        print(f"  [pred={pred_cls}] {X_test[i][:140]}")

## Section 6: Side-by-side summary and recommendation

In [ ]:
lr_report = classification_report(y_test, y_pred_lr, output_dict=True, zero_division=0)
nn_report = classification_report(y_true_nn, y_pred_nn, target_names=target_names, output_dict=True, zero_division=0)

summary = pd.DataFrame({
    'LogReg (TF-IDF)':   [lr_report['accuracy'],
                          lr_report['macro avg']['f1-score'],
                          lr_report['weighted avg']['f1-score']],
    'PyTorch Embedding': [nn_report['accuracy'],
                          nn_report['macro avg']['f1-score'],
                          nn_report['weighted avg']['f1-score']],
}, index=['accuracy', 'macro-F1', 'weighted-F1']).round(3)

print(summary.to_string())

### Which one ships?

On this 4-class setup with ~100K tweets the two models are usually within a couple of macro-F1 points of each other. A few practical considerations:

| Axis | LogReg (TF-IDF) | PyTorch Embedding MLP |
|------|-----------------|-----------------------|
| Train time | ~seconds | ~minutes on CPU, seconds on GPU |
| Inference latency | Microseconds per tweet | A millisecond or so per tweet |
| Memory | The TF-IDF matrix is sparse but the vocab can be big | Small: one 64-dim vector per word |
| Interpretability | Per-class top words via `coef_` | Needs more work (probing, attention, etc.) |
| Improvability | TF-IDF already near its ceiling | Big headroom via pretrained embeddings (Module 2) |

**Recommendation.** For this exact setup, ship LogReg as v1: it's fast to train, easy to explain to the support-ops team, and within noise of the neural model. Keep the PyTorch model as the platform to iterate on. In Module 2 we initialize its `nn.Embedding` with pretrained GloVe vectors, which is when it tends to pull ahead.

## Self-check

1. If you had 1K labeled tweets instead of 100K, which model would you prefer? (LogReg. Neural nets need data.)
2. Why does `padding_idx` matter on `nn.Embedding`? (Keeps padding embedding at zero and blocks gradient flow into it.)
3. What is the cost of routing an Apple tweet to Amazon? (Small: a human reroutes it. Compare to a model that predicts "spam" on a real request, which is costly.)

Module 2 covers dense word embeddings, which pick up exactly where this capstone leaves off.